# wmtMHW paper: Initial eulerian detection
Contains code for figures 2 and 5

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import dask
import cftime

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import AutoDateLocator
import matplotlib.gridspec as gridspec
import seaborn as sns
import cmocean

import cartopy.crs as ccrs

import matplotlib.cm as cm
from matplotlib.patches import Rectangle
from datetime import datetime
import nc_time_axis
from nc_time_axis import AutoCFTimeFormatter, NetCDFTimeDateLocator

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Update these paths for your local setup
cm4_data = "/dfs9/hfdrake_hpc/datasets/CM4_MHW_blobs/"  # old: /pub/hfdrake/datasets/CM4_MHW_blobs/
manso_data = "/pub/mariant3/WarmWaterMasses/data/"

labels = xr.open_dataset(
    f"{manso_data}ocetracv9/ocetrac-v9-blobs-tos-t1-r1-msq0-01860315-01891214-region.nc"
).sel(time=slice("0186", "0186"))

ds = xr.open_dataset(
    f"{cm4_data}data/ocean_daily_cmip.01860101-01901231.tos.nc",
    chunks={"time": 100}
)

region_tos = ds["tos"].sel(
    xh=slice(labels.xh.min(), labels.xh.max()),
    yh=slice(labels.yh.min(), labels.yh.max())
)

static = xr.open_dataset(
    f"{cm4_data}data/WMT_monthly/ocean_month_rho2.static.nc"
)

# Load climatology and percentiles
# clim = xr.open_dataarray(
#     f"{manso_data}/climatology/climatology-manso-0186-01-01-0189-12-31.nc"
# )
# per95 = xr.open_dataset(
#     f"{manso_data}/per95-manso-0186-01-01-0189-12-31.nc"
# )
# max = xr.open_dataset(
#     f"{manso_data}/climatology/climatology-max-manso-tos-0186-01-01-0189-12-31.nc"
# )

In [ ]:
ds = xr.merge([ds, static], join="inner")

clim_max = clim.max(["xh","yh"]).load()
region_mean = region_tos.mean(["xh","yh"]).load()
clim_mean = clim.mean(["xh","yh"]).load()
per95_mean = per95.tos.mean(["xh","yh"]).load()
max_mean = max.tos.mean(["xh","yh"]).load()

region_tos_max = region_tos.max(["xh","yh"]).load()
region_tos_min = region_tos.min(["xh","yh"]).load()

max_time = clim.where(clim == clim.max().item(), drop=True).time[0].values

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.set_facecolor("#ececd7")

plot = clim.sel(time=max_time).plot(cmap=cmocean.cm.thermal, ax=ax)
plot.colorbar.set_label("SST [°C]")
plot.colorbar.set_ticks(np.arange(14, 33, 2))

ax.plot(-87, 29, "c*", markersize=17, markeredgecolor="k", label="Example location")
ax.legend(loc="lower left", fontsize=12)
ax.set(xlabel="", ylabel="", title="")

fig.tight_layout()
# fig.savefig("../figures/warmest-day-sst.png", dpi=350, bbox_inches="tight")
plt.show()

In [ ]:
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import cmocean

xh, yh = -87, 29
line_kwargs = {"alpha": 1, "linewidth": 3}

fig = plt.figure(figsize=(20, 10))
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1])

def fmt_time_axis(ax, locator_kwargs={"max_n_ticks": 8, "calendar": "noleap"}):
    locator = NetCDFTimeDateLocator(**locator_kwargs)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(AutoCFTimeFormatter(locator=locator, calendar="noleap"))

def style_ax(ax, label):
    ax.set_xlim(clim.time[0].values, clim.time[-1].values)
    ax.set_ylim(19, 33)
    ax.set_yticks(np.arange(19, 34, 2))
    ax.set_ylabel("SST [°C]")
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="y", pad=5, width=3, length=5)
    ax.text(0.035, 0.97, label, transform=ax.transAxes, va="top", ha="right", fontweight="bold")
    fmt_time_axis(ax)

ax1 = fig.add_subplot(gs[0])
region_mean.plot(color="k",          label=r"$\mathregular{\overline{T}}_{\mathregular{MANSO}}$", **line_kwargs)
clim_max.plot(color="C4",            label=r"$\langle \mathregular{T}_{\mathregular{max}} \rangle_{\mathregular{MANSO}}$", **line_kwargs)
clim_mean.plot(color="dodgerblue",   label=r"$\langle \mathregular{\overline{T}} \rangle_{\mathregular{MANSO}}$", **line_kwargs)
per95_mean.plot(color="C1",          label=r"$\langle \mathregular{\overline{T}}_{\mathregular{95}} \rangle_{\mathregular{MANSO}}$", **line_kwargs)
ax1.axhline(y=29, color="C5", linestyle="--", linewidth=3, label="29°C isotherm")
ax1.legend(bbox_to_anchor=(1, 0.5), loc="center left", frameon=False)
ax1.tick_params(axis="x", bottom=False, top=False, labelbottom=False)
style_ax(ax1, "(a)")

ax2 = fig.add_subplot(gs[1])
local = region_tos.sel(xh=xh, yh=yh, method="nearest")
ax2.fill_between(local.time.values, 29, local.values, where=(local.values > 29), color="red", alpha=0.5)
local.plot(ax=ax2, color="k",                                                    label=r"$T$", **line_kwargs)
max.tos.sel(xh=xh, yh=yh, method="nearest").plot(ax=ax2, color="C4",            label=r"$\mathregular{T}_{\mathregular{max}}$", **line_kwargs)
clim.sel(xh=xh, yh=yh, method="nearest").plot(ax=ax2, color="dodgerblue",       label=r"$\langle \mathregular{T} \rangle$", **line_kwargs)
per95.tos.sel(xh=xh, yh=yh, method="nearest").plot(ax=ax2, color="C1",          label=r"$\mathregular{T}_{\mathregular{95}}$", **line_kwargs)
ax2.axhline(y=29, color="C5", linestyle="--", linewidth=3, label="29°C isotherm")

handles, labels = ax2.get_legend_handles_labels()
handles.append(Patch(color="red", alpha=0.5))
labels.append("MHW event based on\na 29°C fixed threshold")
ax2.legend(handles, labels, bbox_to_anchor=(1, 0.5), loc="center left", frameon=False)
ax2.set_title("")
ax2.tick_params(axis="x", pad=25, width=3, length=5)
style_ax(ax2, "(b)")

plt.tight_layout()
plt.subplots_adjust(right=0.95, hspace=0.2)
# plt.savefig("../figures/climatologies.png", dpi=350, bbox_inches="tight")
plt.show()